# Stage 07 — Preference pairs

**Track A (Buse) · Stage 7 of 10**

| | |
|---|---|
| **Input** | `data/judge_scores.jsonl` from Sude's relevance judge |
| **Output** | `data/preference_pairs_B.jsonl` |
| **Promotes to** | `src/research_assistant/reranker/pairs_B.py`, `scripts/build_pairs_B.py` |
| **Config** | `configs/reranker_B.yaml` → `pairs:` |

## The handoff

Sude's judge scores retrieved chunks for relevance on a 1-5 scale. You turn those
scores into `(query, chosen, rejected)` triples. Her output schema is your input
contract, and it lives in `src/research_assistant/contracts/judge_J.py`, a joint file.
Pin it before she writes the judge, otherwise this notebook gets rewritten twice.

## Design choices

| Choice | Picked | Alternatives | Why |
|---|---|---|---|
| Pair source | Chunks from the same query's candidate set | Random chunks from the corpus | A pair of "relevant" against "random other paper" is trivially separable and teaches the model nothing useful. Hard negatives from the same candidate set are the whole point. |
| Minimum score gap | 2 on the 1-5 scale | 1, 3 | A gap of 1 is inside the judge's own noise, so those pairs train the model on the judge's jitter. A gap of 3 leaves too few pairs. |
| Pairs per query | Capped at 4 | Uncapped | Without a cap, a handful of queries with large candidate sets dominate the loss. |
| Query source | Unlabelled queries only | Reuse the eval queries | Reusing them leaks the test set into training, which makes every subsequent number a lie. Enforced by `tests/test_no_eval_leakage_J.py`. |
| Judge validation | Required before use | Trust the judge | Ask Sude for the judge's agreement rate against your stage 05 hand labels. Below roughly 0.7 correlation, these pairs are noise wearing a lab coat, and training on them makes the reranker worse in a way the gate will catch late. |

In [ ]:
from _nbsetup_B import REPO, load_cfg, resolve
import json, itertools, collections
from pathlib import Path

pcfg = load_cfg("reranker")["pairs"]

# Sude's judge output. One row per (query, chunk) with a 1-5 relevance score.
scores = [json.loads(l) for l in resolve(pcfg["judge_scores_path"])
          .read_text(encoding="utf-8").splitlines() if l.strip()]
print(len(scores), "judged (query, chunk) rows")
print("schema:", sorted(scores[0].keys()))

In [ ]:
# Leakage guard, run before anything else. The eval queries must not appear here.
eval_queries = {json.loads(l)["query"].strip().lower()
                for l in resolve(pcfg["eval_query_blocklist"]).read_text(encoding="utf-8").splitlines()
                if l.strip()}
leaked = {s["query"].strip().lower() for s in scores} & eval_queries
assert not leaked, f"eval queries leaked into training data: {list(leaked)[:5]}"
print("no leakage:", len(eval_queries), "held-out queries kept clean")

In [ ]:
by_query = collections.defaultdict(list)
for s in scores:
    by_query[s["query"]].append(s)

pairs = []
for query, rows in by_query.items():
    rows = sorted(rows, key=lambda r: -r["score"])
    made = 0
    for hi, lo in itertools.combinations(rows, 2):
        if made >= pcfg["max_pairs_per_query"]:
            break
        if hi["score"] - lo["score"] < pcfg["min_score_gap"]:
            continue
        pairs.append(dict(query=query, chosen=hi["chunk_text"], rejected=lo["chunk_text"],
                          chosen_id=hi["chunk_id"], rejected_id=lo["chunk_id"],
                          score_gap=hi["score"] - lo["score"]))
        made += 1

print(len(pairs), "pairs from", len(by_query), "queries")
import pandas as pd
pd.DataFrame(pairs)["score_gap"].value_counts().sort_index()

In [ ]:
if pcfg["dedupe"]:
    seen, deduped = set(), []
    for p in pairs:
        key = (p["query"], p["chosen_id"], p["rejected_id"])
        if key not in seen:
            seen.add(key); deduped.append(p)
    pairs = deduped

out = resolve(pcfg["out_path"])
with out.open("w", encoding="utf-8") as f:
    for p in pairs:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")
print("wrote", len(pairs), "pairs to", out)

### Sanity read

Print ten pairs and read them. If you cannot see why the chosen chunk beats the
rejected one, the model will not either, and you have found a judge problem rather
than a training problem. This five-minute read is the cheapest bug-catch in the track.

In [ ]:
for p in pairs[:5]:
    print("Q:", p["query"])
    print("  CHOSEN  :", p["chosen"][:160].replace(chr(10), " "))
    print("  REJECTED:", p["rejected"][:160].replace(chr(10), " "))
    print("  gap:", p["score_gap"], "\n")

## Exit checks

- [ ] The leakage assertion passes, and you have seen it fail on purpose once.
- [ ] At least a few hundred pairs. Far fewer and stage 08 will not move the metric.
- [ ] Every rejected chunk came from the same candidate set as its chosen chunk.
- [ ] You read ten pairs and agreed with the judge on at least eight.
- [ ] You know the judge's agreement rate against your stage 05 labels. Ask Sude,
      and record it in the ledger. It is the ceiling on what training can achieve.